# AXI test 9: keep or drop the interior tokens when the state is THOUSANDS of tokens long? (Qwen2.5-7B-Instruct, free Colab T4)
The first KV test (`axi_kv_drop.ipynb`) used graphs of a few hundred tokens. This one uses a graph of about 120 services and 200 edges, a prompt of about **4,000 tokens**. Cutting the tokens of a deleted service out of the model's memory (its KV cache) saves reading time, but the tokens behind the cut were computed while the deleted service was still visible, and with a big graph nearly every token is behind some cut. Does the model write the same reply as when everything is read again from nothing? This notebook measures **exactness, tokens read and kept, and seconds per step**, for four ways of holding the cache (`fresh`, `exact`, `drop_compact`, `drop_gap`).

**The predictions and the reading rules were written and locked BEFORE this run** in `docs/KV_LARGE_TEST.md` (16 lines, each with my probability). Nothing in this notebook changes them. A line that comes out NO is a finding, not an error.

**What to know**
* **Cost: $0.** A local open model, no key, no paid service. It needs the **free T4 GPU**: Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save.
* **Time (my estimate, not measured):** Run A 45 to 70 minutes, Run C the same, Run X (optional, the biggest graph, about 8,000 tokens) 30 to 45 minutes. **Cell 7 measures your machine and tells you the real projection before you commit to the long runs.** You do not have to do everything in one sitting.
* **If Colab disconnects:** reconnect, run Cells 2 to 6 again, then run the run-cell that stopped again. Each finished episode is saved to your Google Drive, so it continues where it stopped.
* **Cell 6 is a safety check on the real model.** If it says STOP, do not run the cells after it: send the printout.
* **Do not paste any key or password anywhere.** This notebook needs none.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-kv-large.zip` (this one, NOT the zip of the first notebook)

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and results are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_kv_large")

### Cell 5: load the model (downloads about 15 GB once, unless this session already did it for the first notebook; about 5 minutes)

In [ ]:
import json, time, torch
from google.colab import files
from axi.experiments.kv_drop import KV_MODEL, preflight, print_verdict_kv
from axi.experiments.kv_large import (LARGE_SEED, LARGE_C_SEED, XL_SEED, LARGE_A_EPISODES, LARGE_A_STEPS, LARGE_C_EPISODES, LARGE_C_STEPS, XL_EPISODES, XL_STEPS,
                                      large_script, xl_script, run_large, print_large, verdict_large)
from axi.experiments.kv_torch import build_kv, make_hf_kit

engine, tok, table, model = build_kv(KV_MODEL, True, "cuda")
kit_plain = make_hf_kit(tok, hint_rules=False)      # the system prompt of every earlier 7B test (Run A)
kit_rule = make_hf_kit(tok, hint_rules=True)        # the same prompt with the rule stated (Runs C and X)
T0 = time.time()
def prog(i, n): print(f"  episode {i}/{n}   {time.time() - T0:6.0f} s", end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
res_a = res_c = res_x = None
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1), "| model:", KV_MODEL)

### Cell 6: SAFETY CHECK on the real model, on the big prompt (about 2 minutes). It prints one line per check and stops if any fails. If it says STOP, do not run the next cells: send me the printout

In [ ]:
checks = preflight(engine, kit_plain, LARGE_SEED, large_script)
for name, ok, detail in checks: print(("ok    " if ok else "FAIL  ") + name + "   [" + detail + "]")
json.dump([dict(name=n, ok=bool(o), detail=d) for n, o, d in checks], open("/content/out/axi_large_preflight.json", "w"), indent=1); store.persist("axi_large_preflight.json")
assert all(ok for _, ok, _ in checks), "STOP: a check failed. Do not run the next cells. Send the printout above."
print("\nALL CHECKS OK. Go on to Cell 7.")

### Cell 7: quick check, 1 episode of 3 steps with all four ways (about 2 to 4 minutes). It also tells you how long the long runs will take on this machine. If it fails, stop and send me the red error text

In [ ]:
smoke = run_large(engine, kit_plain, table, 1, 3, 99, False, "/content/smoke_large.jsonl", large_script, meta=dict(model=KV_MODEL, smoke=True))
print_large(smoke["summary"], "smoke, 1 episode x 3 steps")
step_s = sum(v["seconds"] for v in smoke["summary"].values() if v.get("n"))
print(f"\nOne step of all four ways took {step_s:.1f} s on this machine (the first steps include warm-up, so the long runs are probably a little quicker).")
print(f"Projection: Run A ({LARGE_A_EPISODES * LARGE_A_STEPS} steps) about {LARGE_A_EPISODES * LARGE_A_STEPS * step_s / 60:.0f} minutes; Run C about the same; Run X ({XL_EPISODES * XL_STEPS} steps on a graph twice as big, roughly 2 to 2.5 times the time per step) about {XL_EPISODES * XL_STEPS * step_s * 2.25 / 60:.0f} minutes.")

### Cell 8: RUN A, no rule. 4 episodes x 40 steps, all four ways (the counter shows episode i/4; each episode takes about a quarter of the projection from Cell 7)

In [ ]:
T0 = time.time(); name = "axi_large_a"
print("progress file:", store.restore(name + ".jsonl"))
res_a = run_large(engine, kit_plain, table, LARGE_A_EPISODES, LARGE_A_STEPS, LARGE_SEED, False, store.path(name + ".jsonl"), large_script, meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog))
print(); print_large(res_a["summary"], "RUN A: no rule")
print_verdict_kv(verdict_large(res_a["summary"]), "RUN A (lines 14 and 16 need Runs C and X)")
files.download(save(res_a, store.path(name + ".json"))); store.persist(name + ".json")
torch.cuda.empty_cache()

### Cell 9: RUN C, the rule stated in the prompt. The same 4 request scripts

In [ ]:
assert res_a, "Run Cell 8 first."
T0 = time.time(); name = "axi_large_c"
print("progress file:", store.restore(name + ".jsonl"))
res_c = run_large(engine, kit_rule, table, LARGE_C_EPISODES, LARGE_C_STEPS, LARGE_C_SEED, True, store.path(name + ".jsonl"), large_script, meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog))
print(); print_large(res_c["summary"], "RUN C: rule stated")
print_verdict_kv(verdict_large(res_a["summary"], res_c["summary"]), "RUN A and RUN C (line 16 needs Run X)")
files.download(save(res_c, store.path(name + ".json"))); store.persist(name + ".json")
torch.cuda.empty_cache()

### Cell 10 (optional): RUN X, the biggest graph (240 services, 400 edges, about 8,000 tokens). 2 episodes x 30 steps, rule stated, a different seed. If the GPU runs out of memory the cell stops with an error: that is itself a result, send me the red text

In [ ]:
assert res_a and res_c, "Run Cells 8 and 9 first."
T0 = time.time(); name = "axi_large_x"
print("progress file:", store.restore(name + ".jsonl"))
res_x = run_large(engine, kit_rule, table, XL_EPISODES, XL_STEPS, XL_SEED, True, store.path(name + ".jsonl"), xl_script, meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog))
print(); print_large(res_x["summary"], "RUN X: about 8,000 tokens, rule stated")
files.download(save(res_x, store.path(name + ".json"))); store.persist(name + ".json")
torch.cuda.empty_cache()

### Cell 11: the sixteen reading rules, all at once (prints 'not run' for a run you skipped)

In [ ]:
assert res_a, "Run Cell 8 first."
final = verdict_large(res_a["summary"], res_c["summary"] if res_c else None, res_x["summary"] if res_x else None)
print_verdict_kv(final, "ALL RUNS")
print("peak GPU memory (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 1), "| minutes since the last run started:", round((time.time() - T0) / 60, 1))

### Done
Files downloaded: `axi_large_a.json`, `axi_large_c.json` (and `axi_large_x.json` if you ran Cell 10); the checks are also in `axi_large_preflight.json` on your Drive (folder `axi_kv_large`). Upload the JSON files to the chat, with a screenshot of the printed tables and of the sixteen reading-rule lines. Do not paste any keys anywhere.

When the numbers are in, `docs/KV_LARGE_TEST.md` gets a "Result" section **below** the locked predictions; nothing above it changes.